# 01 · Source quality and scope

**Question:** Is the recorded session dataset sufficiently consistent for descriptive conversion analysis? The unit of observation is a session, not an identified individual. Original records are retained.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
SOURCE = ROOT / "data" / "raw" / "online_shoppers_intention.csv"
if not SOURCE.is_file():
    raise FileNotFoundError("Run this notebook from the repository root after downloading the UCI CSV")
data = pd.read_csv(SOURCE)
print(f"Loaded {len(data):,} recorded sessions")

In [ ]:
import hashlib
from scripts.audit_sessions import audit

digest = hashlib.sha256(SOURCE.read_bytes()).hexdigest().upper()
metrics, profile = audit(data, digest)
display(profile)
print(f"Purchases: {metrics['purchase_sessions']:,} / {metrics['sessions']:,} ({metrics['conversion_rate_pct']:.3f}%)")
print(f"Missing cells: {metrics['missing_cells']:,}; later exact-match rows: {metrics['exact_duplicate_rows']:,}")
print(f"Deduplication sensitivity: {metrics['duplicate_sensitivity_pp']:+.3f} percentage points")

**Interpretation:** Matching feature rows do not prove duplicate shopping events. The primary analysis retains all records; the deduplicated conversion rate is a sensitivity check. Recorded months cannot establish a full annual seasonal pattern.

In [ ]:
display(data.groupby('VisitorType', dropna=False)['Revenue'].agg(sessions='size', purchase_sessions='sum'))
print('Observed months:', ', '.join(metrics['months_observed']))

**Next:** Explore visitor and engagement differences in Notebook 02. For a reproducible written data-quality summary, run `python scripts/audit_sessions.py`.

## Upgrade checkpoint: preserve source integrity

The advanced analysis retains every source row, compares group counts against the audit, and excludes transaction-linked `PageValues` from its principal models.

In [ ]:
assert metrics["sessions"] == len(data)
assert metrics["purchase_sessions"] + metrics["nonpurchase_sessions"] == len(data)
print("Audit reconciliation passed; duplicate sensitivity (pp):", metrics["duplicate_sensitivity_pp"])